# PDM Score One-Stage Evaluation

This notebook reproduces `run_pdm_score_one_stage.py` interactively on the `mini` split.

**Steps:**
1. Setup paths and instantiate components
2. Load scenes and metric cache
3. Run per-scenario PDM scoring
4. Aggregate two-frame comfort and compute final scores
5. Visualize results

## 1. Config

In [ ]:
import os
from pathlib import Path

# --- Adjust these paths if needed ---
NAVSIM_ROOT       = Path("/home/weiyu/workspace/navsim")
OPENSCENE_DATA_ROOT = NAVSIM_ROOT / "dataset"
NAVSIM_EXP_ROOT   = NAVSIM_ROOT / "exp"

DATA_SPLIT        = "mini"
MAX_SCENES        = 10          # set None to run all
TRAFFIC_AGENTS    = "non_reactive"   # "non_reactive" or "reactive"

NAVSIM_LOG_PATH        = OPENSCENE_DATA_ROOT / "navsim_logs" / DATA_SPLIT
ORIGINAL_SENSOR_PATH   = OPENSCENE_DATA_ROOT / "sensor_blobs" / DATA_SPLIT
METRIC_CACHE_PATH      = NAVSIM_EXP_ROOT / "metric_cache"
OUTPUT_DIR             = NAVSIM_EXP_ROOT / "pdm_score_one_stage_notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Log path      : {NAVSIM_LOG_PATH}")
print(f"Metric cache  : {METRIC_CACHE_PATH}")
print(f"Output dir    : {OUTPUT_DIR}")

## 2. Instantiate Simulator, Scorer, Agent, Traffic Policy

In [ ]:
from nuplan.planning.simulation.trajectory.trajectory_sampling import TrajectorySampling

from navsim.agents.constant_velocity_agent import ConstantVelocityAgent
from navsim.planning.simulation.planner.pdm_planner.simulation.pdm_simulator import PDMSimulator
from navsim.planning.simulation.planner.pdm_planner.scoring.pdm_scorer import PDMScorer, PDMScorerConfig
from navsim.traffic_agents_policies.log_replay_traffic_agents import LogReplayTrafficAgents
from navsim.traffic_agents_policies.navsim_IDM_traffic_agents import NavsimIDMTrafficAgents
from navsim.planning.simulation.observation.navsim_idm_agents import NavsimIDMAgents

proposal_sampling = TrajectorySampling(num_poses=40, interval_length=0.1)

simulator = PDMSimulator(proposal_sampling=proposal_sampling)

scorer = PDMScorer(
    proposal_sampling=proposal_sampling,
    config=PDMScorerConfig(
        progress_weight=5.0,
        ttc_weight=5.0,
        lane_keeping_weight=2.0,
        history_comfort_weight=2.0,
        two_frame_extended_comfort_weight=2.0,
    ),
)

agent = ConstantVelocityAgent()
agent.initialize()

if TRAFFIC_AGENTS == "non_reactive":
    traffic_agents_policy = LogReplayTrafficAgents(future_trajectory_sampling=proposal_sampling)
else:
    idm_obs = NavsimIDMAgents(
        target_velocity=10, min_gap_to_lead_agent=1.0, headway_time=1.5,
        accel_max=1.0, decel_max=2.0, open_loop_detections_types=[],
        minimum_path_length=20, planned_trajectory_samples=None,
        planned_trajectory_sample_interval=None, radius=100,
        add_open_loop_parked_vehicles=True, idm_snap_threshold=3.0,
    )
    traffic_agents_policy = NavsimIDMTrafficAgents(
        future_trajectory_sampling=proposal_sampling, idm_agents_observation=idm_obs
    )

print(f"Agent         : {agent.__class__.__name__}")
print(f"Traffic policy: {traffic_agents_policy.__class__.__name__}")
print(f"Proposal steps: {proposal_sampling.num_poses} x {proposal_sampling.interval_length}s")

## 3. Load Scenes and Metric Cache

In [ ]:
from navsim.common.dataclasses import SceneFilter, SensorConfig
from navsim.common.dataloader import MetricCacheLoader, SceneLoader

scene_filter = SceneFilter(
    num_history_frames=4,
    num_future_frames=10,
    has_route=True,
    max_scenes=MAX_SCENES,
)

# Lightweight loader just to get token list
scene_loader_tokens = SceneLoader(
    original_sensor_path=None,
    data_path=NAVSIM_LOG_PATH,
    scene_filter=scene_filter,
    sensor_config=SensorConfig.build_no_sensors(),
)

metric_cache_loader = MetricCacheLoader(METRIC_CACHE_PATH)

tokens_to_evaluate = list(set(scene_loader_tokens.tokens) & set(metric_cache_loader.tokens))
print(f"Scenes in split          : {len(scene_loader_tokens.tokens)}")
print(f"Metric cache entries     : {len(metric_cache_loader.tokens)}")
print(f"Tokens to evaluate       : {len(tokens_to_evaluate)}")

## 4. Run Per-Scenario PDM Scoring

In [ ]:
import traceback
import pandas as pd
from nuplan.common.actor_state.state_representation import StateSE2
from nuplan.common.geometry.convert import relative_to_absolute_poses

from navsim.common.dataclasses import PDMResults
from navsim.evaluate.pdm_score import pdm_score

# Full loader with sensor config from agent
scene_loader = SceneLoader(
    original_sensor_path=ORIGINAL_SENSOR_PATH,
    data_path=NAVSIM_LOG_PATH,
    scene_filter=scene_filter,
    sensor_config=agent.get_sensor_config(),
)

pdm_results = []

for idx, token in enumerate(tokens_to_evaluate):
    print(f"[{idx+1}/{len(tokens_to_evaluate)}] token={token}")
    try:
        metric_cache = metric_cache_loader.get_from_token(token)
        agent_input  = scene_loader.get_agent_input_from_token(token)

        if agent.requires_scene:
            scene      = scene_loader.get_scene_from_token(token)
            trajectory = agent.compute_trajectory(agent_input, scene)
        else:
            trajectory = agent.compute_trajectory(agent_input)

        score_row, ego_simulated_states = pdm_score(
            metric_cache=metric_cache,
            model_trajectory=trajectory,
            future_sampling=simulator.proposal_sampling,
            simulator=simulator,
            scorer=scorer,
            traffic_agents_policy=traffic_agents_policy,
        )

        end_pose = StateSE2(
            x=trajectory.poses[-1, 0],
            y=trajectory.poses[-1, 1],
            heading=trajectory.poses[-1, 2],
        )
        absolute_endpoint = relative_to_absolute_poses(metric_cache.ego_state.rear_axle, [end_pose])[0]

        score_row["valid"]          = True
        score_row["log_name"]       = metric_cache.log_name
        score_row["frame_type"]     = metric_cache.scene_type
        score_row["start_time"]     = metric_cache.timepoint.time_s
        score_row["endpoint_x"]     = absolute_endpoint.x
        score_row["endpoint_y"]     = absolute_endpoint.y
        score_row["start_point_x"]  = metric_cache.ego_state.rear_axle.x
        score_row["start_point_y"]  = metric_cache.ego_state.rear_axle.y
        score_row["ego_simulated_states"] = [ego_simulated_states]

    except Exception:
        print(f"  WARNING: agent failed for token {token}")
        traceback.print_exc()
        score_row = pd.DataFrame([PDMResults.get_empty_results()])
        score_row["valid"] = False

    score_row["token"] = token
    pdm_results.append(score_row)

pdm_score_df = pd.concat(pdm_results, ignore_index=True)
print(f"\nDone. Shape: {pdm_score_df.shape}")

## 5. Aggregate Two-Frame Comfort and Compute Final Scores

In [ ]:
import numpy as np
from navsim.common.enums import SceneFrameType
from navsim.planning.simulation.planner.pdm_planner.scoring.scene_aggregator import SceneAggregator
from navsim.planning.simulation.planner.pdm_planner.utils.pdm_enums import WeightedMetricIndex


def infer_start_adjacent_mapping(score_df, time_gap_threshold=0.55):
    adjacent_mapping = {}
    for log_name, group_df in score_df[score_df["frame_type"] == SceneFrameType.ORIGINAL].groupby("log_name"):
        group_df = group_df.sort_values("start_time").reset_index(drop=True)
        for i in range(1, len(group_df)):
            prev_row    = group_df.iloc[i - 1]
            current_row = group_df.iloc[i]
            if abs(current_row["start_time"] - prev_row["start_time"]) <= time_gap_threshold:
                adjacent_mapping[current_row["token"]] = prev_row["token"]
    return adjacent_mapping


def create_scene_aggregators(all_mappings, full_score_df, proposal_sampling):
    full_score_df["two_frame_extended_comfort"] = np.nan
    full_score_df = full_score_df.set_index("token")
    all_updates = []
    for now_frame, previous_frame in all_mappings.items():
        aggregator = SceneAggregator(
            now_frame=now_frame,
            previous_frame=previous_frame,
            score_df=full_score_df,
            proposal_sampling=proposal_sampling,
        )
        all_updates.append(aggregator.aggregate_scores(one_stage_only=True))
    if all_updates:
        all_updates_df = pd.concat(all_updates, ignore_index=True).set_index("token")
        full_score_df.update(all_updates_df)
    full_score_df.reset_index(inplace=True)
    return full_score_df.drop(columns=["ego_simulated_states"])


def compute_final_scores(pdm_score_df):
    df = pdm_score_df.copy()
    two_frame_scores      = df["two_frame_extended_comfort"].to_numpy()
    weighted_metrics      = np.stack(df["weighted_metrics"].to_numpy())
    weighted_metrics_array = np.stack(df["weighted_metrics_array"].to_numpy())
    mask         = np.isnan(two_frame_scores)
    two_frame_idx = WeightedMetricIndex.TWO_FRAME_EXTENDED_COMFORT
    weighted_metrics[mask, two_frame_idx]       = 0.0
    weighted_metrics_array[mask, two_frame_idx] = 0.0
    weighted_metrics[~mask, two_frame_idx]      = two_frame_scores[~mask]
    weighted_sum  = (weighted_metrics * weighted_metrics_array).sum(axis=1)
    total_weight  = weighted_metrics_array.sum(axis=1)
    total_weight[total_weight == 0.0] = np.nan
    df["score"] = df["multiplicative_metrics_prod"].to_numpy() * (weighted_sum / total_weight)
    df.drop(columns=["weighted_metrics", "weighted_metrics_array", "multiplicative_metrics_prod"], inplace=True)
    return df


adjacent_mapping = infer_start_adjacent_mapping(pdm_score_df)
print(f"Adjacent frame pairs found: {len(adjacent_mapping)}")

pdm_score_df = create_scene_aggregators(adjacent_mapping, pdm_score_df, proposal_sampling)
pdm_score_df = compute_final_scores(pdm_score_df)

print(f"Final df shape: {pdm_score_df.shape}")
pdm_score_df.head()

## 6. Summary and Save Results

In [ ]:
from dataclasses import fields
from datetime import datetime

score_cols = [
    c for c in pdm_score_df.columns
    if (any(f.name in c for f in fields(PDMResults)) or c in ("two_frame_extended_comfort", "score"))
    and c != "pdm_score"
]

average_row = pdm_score_df[score_cols].mean(skipna=True)
average_row["token"] = "average_all_frames"
average_row["valid"] = pdm_score_df["valid"].all()

output_df = pdm_score_df[["token", "valid"] + score_cols].copy()
output_df.loc[len(output_df)] = average_row

timestamp = datetime.now().strftime("%Y.%m.%d.%H.%M.%S")
csv_path  = OUTPUT_DIR / f"{timestamp}.csv"
output_df.to_csv(csv_path, index=False)

num_ok   = int(pdm_score_df["valid"].sum())
num_fail = len(pdm_score_df) - num_ok
print(f"Successful : {num_ok}")
print(f"Failed     : {num_fail}")
print(f"Avg score  : {pdm_score_df['score'].mean():.4f}")
print(f"Saved to   : {csv_path}")

## 7. Visualize: Per-Scenario Scores

In [ ]:
import matplotlib.pyplot as plt

metric_cols = [
    "no_at_fault_collision",
    "drivable_area_compliance",
    "ego_progress",
    "time_to_collision_within_bound",
    "comfort",
    "score",
]
metric_cols = [c for c in metric_cols if c in pdm_score_df.columns]

plot_df = pdm_score_df[["token"] + metric_cols].copy()
plot_df["token"] = plot_df["token"].str[:8]  # shorten for readability
plot_df = plot_df.set_index("token")

fig, ax = plt.subplots(figsize=(max(10, len(plot_df) * 0.8), 4))
plot_df[metric_cols].T.plot(kind="bar", ax=ax)
ax.set_title("Per-Scenario PDM Metrics")
ax.set_ylabel("Score (0–1)")
ax.set_ylim(0, 1.05)
ax.legend(title="token", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=7)
plt.tight_layout()
plt.show()

## 8. Visualize: Average Metric Breakdown

In [ ]:
avg = pdm_score_df[metric_cols].mean(skipna=True)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(avg.index, avg.values, color="steelblue", edgecolor="white")
ax.set_title(f"Average PDM Metrics over {len(pdm_score_df)} scenes")
ax.set_ylabel("Score (0–1)")
ax.set_ylim(0, 1.05)
for bar, val in zip(bars, avg.values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02, f"{val:.3f}",
            ha="center", va="bottom", fontsize=9)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 9. Visualize: BEV with Agent Trajectory

Uses navsim's built-in `plot_bev_with_agent` to show predicted vs. ground-truth trajectory for each evaluated scene.

In [ ]:
from navsim.visualization.plots import plot_bev_with_agent

# Reload scene_loader with camera sensor config if needed for BEV
scene_loader_bev = SceneLoader(
    original_sensor_path=ORIGINAL_SENSOR_PATH,
    data_path=NAVSIM_LOG_PATH,
    scene_filter=scene_filter,
    sensor_config=agent.get_sensor_config(),
)

n_cols = 2
n_rows = (len(tokens_to_evaluate) + 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 5 * n_rows))
axes = axes.flatten()

for i, token in enumerate(tokens_to_evaluate):
    scene = scene_loader_bev.get_scene_from_token(token)
    f, a  = plot_bev_with_agent(scene, agent)

    row = pdm_score_df[pdm_score_df["token"] == token]
    sc  = row["score"].values[0] if not row.empty else float("nan")
    a.set_title(f"{token[:10]}...  score={sc:.3f}")

    # Copy rendered figure into the grid
    f.canvas.draw()
    import numpy as np
    img = np.frombuffer(f.canvas.tostring_rgb(), dtype=np.uint8)
    img = img.reshape(f.canvas.get_width_height()[::-1] + (3,))
    axes[i].imshow(img)
    axes[i].axis("off")
    plt.close(f)

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle("BEV: Agent Trajectory vs Ground Truth", fontsize=14)
plt.tight_layout()
plt.show()